### Transformations Module — Usage Examples

This guide demonstrates how to use the transformations utilities from the pyspark_utils library.
These helpers simplify common data engineering tasks such as:
- Flattening nested structures
- Exploding arrays
- Joining datasets
- Aggregating metrics
- Hashing sensitive fields
- Pivoting tables
- Using window functions for ranking, rolling totals, and time‑based logic

### Import Libraries
-----

In [ ]:
from pyspark.sql import SparkSession
from pyspark_utils.cleaning import (
    column_names,
    nulls,
    deduplication,
    dates,
    type_casting,
    text_cleaning
)

### Example Dataset
---

In [ ]:
claims = [
    ("POL123", "2024-01-10", 500.0, ["glass", "tow"]),
    ("POL123", "2024-02-15", 1200.0, ["collision"]),
    ("POL456", "2024-01-20", 300.0, ["theft", "investigation"]),
    ("POL456", "2024-03-01", 900.0, ["collision"]),
]
columns = ["policy_number", "claim_date", "claim_amount", "claim_items"]
df = spark.createDataFrame(claims, columns)


### Flattening & Exploding
Flatten a struct or explode an array

In [ ]:
from pyspark_utils.transformations import flatten, arrays

# Explode claim items into separate rows
df_exploded = flatten.explode_array(df, "claim_items")
df_exploded.show()


### Joins

In [ ]:
from pyspark_utils.transformations import joins


# Safe Join
df_joined = joins.safe_join(df_policies, df_claims, keys="policy_number")

# Broadcast join for small lookup tables 
df_broadcast = joins.broadcast_join(df_claims, df_lookup, keys="code")


### Aggregations

In [ ]:
from pyspark_utils.transformations import aggregations

df_agg = aggregations.aggregate_by(
    df,
    group_cols=["policy_number"],
    agg_map={"claim_amount": "sum"}
)
df_agg.show()


### Array Utilities

In [ ]:
from pyspark_utils.transformations import arrays

df_sizes = arrays.array_size(df, "claim_items")
df_unique = arrays.array_unique(df, "claim_items")


### Hashing Sensitive Fields

In [ ]:
from pyspark_utils.transformations import hashing

df_hashed = hashing.hash_columns(df, ["policy_number", "claim_date"])


### Pivoting

In [ ]:
from pyspark_utils.transformations import pivots

df_pivot = pivots.pivot_table(
    df_exploded,
    index=["policy_number"],
    column="claim_items",
    value="claim_amount"
)


### Window Functions
---

**Latest claim per policy**

In [ ]:
from pyspark_utils.transformations import window

df_latest = window.latest_record(
    df,
    keys=["policy_number"],
    order_by="claim_date"
)


**Rolling claim total**

In [ ]:
df_roll = window.rolling_sum(
    df,
    partition_cols=["policy_number"],
    order_col="claim_date",
    target_col="claim_amount"
)


**Lag for fraud detection**

In [ ]:
df_lag = window.add_lag(
    df,
    column="claim_amount",
    offset=1,
    partition_cols=["policy_number"],
    order_col="claim_date"
)
